# Hospital Readmission Prediction — Diabetes 130-US Hospitals

**Objective:** Predict whether a diabetic patient will be readmitted to the hospital within 30 days of discharge.  
**Approach:** Train Logistic Regression, Random Forest, and XGBoost classifiers on the same preprocessed data, evaluate all three, select the best-performing and most stable model, then apply Permutation Feature Importance (PFI) and Partial Dependence Plots (PDP) for explainability.

---

### Table of Contents
1. Setup & Imports
2. Data Loading & Inspection
3. Data Cleaning
4. Feature Engineering & Binary Target
5. Exploratory Data Analysis (EDA)
6. Statistical Tests
7. Preprocessing (Encoding, Splitting, SMOTE, Scaling)
8. Model Training (LR, RF, XGBoost)
9. Model Evaluation & Comparison
10. Best Model Selection
11. Explainability — Permutation Feature Importance (PFI)
12. Explainability — Partial Dependence Plots (PDP)
13. Classification Threshold Analysis
14. Conclusion

---
## 1. Setup & Imports

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

from src.ds_utils import (
    profile,
    detect_outliers_iqr,
    plot_distributions,
    plot_correlation_matrix,
    plot_categorical_counts,
    plot_target_distribution,
    plot_target_vs_features,
    clean_diabetes_data,
    engineer_features,
    create_binary_target,
    encode_and_prepare,
    evaluate_classifier,
    plot_confusion_matrices,
    plot_roc_curves,
    plot_precision_recall_curves,
    compare_models,
    quick_cross_val,
    cross_val_box_plot,
    plot_permutation_importance,
    plot_partial_dependence,
    threshold_analysis,
    normality_test,
    correlation_test,
    chi2_test,
)

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline
print('All imports successful!')

---
## 2. Data Loading & Inspection

In [ ]:
df_raw = pd.read_csv('../Data/diabetic_data.csv')
print(f'Dataset shape: {df_raw.shape}')
print(f'Columns ({len(df_raw.columns)}): {df_raw.columns.tolist()}')
df_raw.head()

In [ ]:
print('Data types:')
print(df_raw.dtypes.value_counts())
print(f'\nMemory usage: {df_raw.memory_usage(deep=True).sum() / 1e6:.1f} MB')

In [ ]:
# Full data profile
prof = profile(df_raw)
prof

In [ ]:
# Original target distribution (before binary conversion)
print('Original readmitted distribution:')
print(df_raw['readmitted'].value_counts())
print()
plot_target_distribution(df_raw, 'readmitted')

---
## 3. Data Cleaning

In [ ]:
df = clean_diabetes_data(df_raw)
print(f'\nShape after cleaning: {df.shape}')
print(f'Remaining columns: {df.columns.tolist()}')

In [ ]:
# Check remaining missing values
null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0].sort_values(ascending=False)
if len(null_counts) > 0:
    print('Remaining missing values:')
    print(null_counts)
    print()
    null_counts.plot.bar(color='salmon', edgecolor='black', figsize=(8, 4))
    plt.title('Missing Values After Cleaning')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('No missing values remain!')

---
## 4. Feature Engineering & Binary Target

In [ ]:
df = engineer_features(df)
print(f'Shape after feature engineering: {df.shape}')
df.head()

In [ ]:
# Convert readmitted to binary: 1 = readmitted within 30 days, 0 = otherwise
print('Before binary conversion:')
print(df['readmitted'].value_counts())
print()

df = create_binary_target(df, col='readmitted')

print('After binary conversion (1 = readmitted <30 days, 0 = not):')
print(df['readmitted'].value_counts())
print(f'\nPositive class rate: {df["readmitted"].mean():.2%}')

In [ ]:
# Drop columns no longer needed after feature engineering
drop_cols = [c for c in ['age', 'payer_code'] if c in df.columns]
# Also drop individual medication columns (summarized into num_med_changed/num_med_active)
med_cols = ['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
            'glimepiride', 'glipizide', 'glyburide', 'pioglitazone',
            'rosiglitazone', 'insulin', 'acarbose', 'miglitol',
            'tolbutamide', 'tolazamide', 'troglitazone', 'acetohexamide',
            'glyburide-metformin', 'glipizide-metformin',
            'glimepiride-pioglitazone', 'metformin-rosiglitazone',
            'metformin-pioglitazone']
drop_cols += [c for c in med_cols if c in df.columns]
df.drop(columns=drop_cols, inplace=True)
print(f'Dropped {len(drop_cols)} columns. Shape: {df.shape}')
print(f'Remaining columns: {df.columns.tolist()}')

---
## 5. Exploratory Data Analysis (EDA)

In [ ]:
# Binary target distribution
plot_target_distribution(df, 'readmitted')

In [ ]:
# Numeric feature distributions
num_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
            'num_medications', 'number_diagnoses', 'total_visits',
            'age_numeric', 'num_med_changed', 'num_med_active']
plot_distributions(df, cols=num_cols)

In [ ]:
# Correlation matrix for numeric features
plot_correlation_matrix(df)

In [ ]:
# Categorical feature distributions
cat_cols = [c for c in df.select_dtypes(include='object').columns if c != 'readmitted']
plot_categorical_counts(df, cols=cat_cols[:6], top_n=8)

In [ ]:
# Features vs target (box plots)
key_features = ['time_in_hospital', 'num_lab_procedures', 'num_medications',
                'number_diagnoses', 'total_visits', 'age_numeric']
plot_target_vs_features(df, target='readmitted', cols=key_features)

In [ ]:
# Readmission rate by age group
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# By age
age_readmit = df.groupby('age_numeric')['readmitted'].mean()
age_readmit.plot.bar(ax=axes[0], color='#2196F3', edgecolor='black')
axes[0].set_title('Readmission Rate by Age Group', fontweight='bold')
axes[0].set_ylabel('Readmission Rate')
axes[0].set_xlabel('Age (midpoint)')
axes[0].tick_params(axis='x', rotation=0)

# By number of inpatient visits
inpatient_readmit = df.groupby('number_inpatient')['readmitted'].mean().head(10)
inpatient_readmit.plot.bar(ax=axes[1], color='#FF5722', edgecolor='black')
axes[1].set_title('Readmission Rate by Prior Inpatient Visits', fontweight='bold')
axes[1].set_ylabel('Readmission Rate')
axes[1].set_xlabel('Number of Inpatient Visits')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

---
## 6. Statistical Tests

In [ ]:
# Normality test on key numeric features
print('=== Normality Tests (Shapiro-Wilk) ===')
for col in ['time_in_hospital', 'num_lab_procedures', 'num_medications']:
    print(f'\n{col}:')
    normality_test(df[col])

In [ ]:
# Correlation tests
print('=== Correlation: num_medications vs num_lab_procedures ===')
correlation_test(df['num_medications'], df['num_lab_procedures'])

In [ ]:
# Chi-squared: readmitted vs diagnosis category
print('=== Chi-squared: readmitted vs diag_1 ===')
# Need to use the pre-binary version for chi2
chi2_test(df, 'diag_1', 'readmitted')

print('\n=== Chi-squared: readmitted vs gender ===')
chi2_test(df, 'gender', 'readmitted')

---
## 7. Preprocessing (Encoding, Splitting, SMOTE, Scaling)

Pipeline:
1. One-hot encode categoricals + drop high-cardinality columns
2. Stratified train/test split (80/20)
3. Apply SMOTE **on training set only** to address class imbalance
4. StandardScaler on all features

In [ ]:
# Encode and split
X_train, X_test, y_train, y_test, feature_names = encode_and_prepare(
    df, target='readmitted', test_size=0.2, random_state=42
)
print(f'\nBefore SMOTE:')
print(f'  X_train: {X_train.shape}, y_train distribution: {dict(y_train.value_counts())}')
print(f'  X_test:  {X_test.shape},  y_test distribution:  {dict(y_test.value_counts())}')

In [ ]:
# Apply SMOTE on training data only
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f'After SMOTE:')
print(f'  X_train: {X_train_sm.shape}, y_train distribution: {dict(pd.Series(y_train_sm).value_counts())}')
print(f'  X_test:  {X_test.shape} (unchanged — no SMOTE on test set)')

---
## 8. Model Training

We train three classifiers on the same SMOTE-balanced training data:
- **Logistic Regression** (linear baseline)
- **Random Forest** (ensemble, bagging)
- **XGBoost** (ensemble, boosting)

In [ ]:
# --- Logistic Regression ---
lr = LogisticRegression(max_iter=1000, random_state=42, solver='lbfgs')
lr.fit(X_train_sm, y_train_sm)
print('Logistic Regression trained.')

# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_split=10,
                            random_state=42, n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)
print('Random Forest trained.')

# --- XGBoost ---
xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1,
                     random_state=42, eval_metric='logloss', n_jobs=-1)
xgb.fit(X_train_sm, y_train_sm)
print('XGBoost trained.')

---
## 9. Model Evaluation & Comparison

In [ ]:
# Evaluate each model on the held-out test set
label_names = ['Not Readmitted', 'Readmitted <30d']

results = {}
trained_models = {
    'Logistic Regression': lr,
    'Random Forest': rf,
    'XGBoost': xgb
}

for name, model in trained_models.items():
    print(f'\n{"=" * 50}')
    print(f'  {name}')
    print(f'{"=" * 50}')
    results[name] = evaluate_classifier(model, X_test, y_test, label_names)

In [ ]:
# Side-by-side confusion matrices
plot_confusion_matrices(trained_models, X_test, y_test, label_names)

In [ ]:
# === ROC Curves (all 3 models on one plot) ===
plot_roc_curves(trained_models, X_test, y_test)

In [ ]:
# === Precision-Recall Curves (all 3 models on one plot) ===
plot_precision_recall_curves(trained_models, X_test, y_test)

In [ ]:
# Metric comparison bar chart
df_results = compare_models(results)
df_results

In [ ]:
# Cross-validation stability (box plot)
print('=== 5-Fold Stratified Cross-Validation ===')
cv_scores = cross_val_box_plot(trained_models, X_train_sm, y_train_sm,
                                cv=5, scoring='roc_auc')

---
## 10. Best Model Selection

> *"The best-performing and most stable model will be chosen for the explainability analysis."*

Selection criteria:
- Highest ROC-AUC on the test set
- Lowest CV standard deviation (stability)
- Best F1 score

In [ ]:
# Build a selection summary
selection = pd.DataFrame({
    'Test ROC-AUC': {name: results[name]['roc_auc'] for name in trained_models},
    'Test F1': {name: results[name]['f1'] for name in trained_models},
    'CV ROC-AUC Mean': {name: cv_scores[name].mean() for name in trained_models},
    'CV ROC-AUC Std': {name: cv_scores[name].std() for name in trained_models},
})
selection['Stability (1 - Std)'] = 1 - selection['CV ROC-AUC Std']

print('=== Model Selection Summary ===')
print(selection.round(4).to_string())

# Select the model with the best ROC-AUC
best_name = selection['Test ROC-AUC'].idxmax()
best_model = trained_models[best_name]
print(f'\n>>> Best model selected: {best_name} (Test ROC-AUC = {selection.loc[best_name, "Test ROC-AUC"]:.4f})')

---
## 11. Explainability — Permutation Feature Importance (PFI)

PFI measures how much the model's ROC-AUC drops when each feature is randomly shuffled.  
Applied **only on the selected best model**.

In [ ]:
print(f'Computing PFI for: {best_name}')
pfi_results = plot_permutation_importance(
    best_model, X_test, y_test,
    feature_names=feature_names,
    top_n=15, n_repeats=10, scoring='roc_auc'
)
print('\nTop 10 most important features:')
pfi_results.head(10)

---
## 12. Explainability — Partial Dependence Plots (PDP)

PDP shows the marginal effect of a feature on the predicted probability.  
We plot the top features identified by PFI.

In [ ]:
# Get the top features from PFI (only numeric / meaningful ones)
top_features = pfi_results['feature'].head(6).tolist()

# Find their indices in feature_names
feature_indices = [feature_names.index(f) for f in top_features if f in feature_names]

print(f'PDP for: {best_name}')
print(f'Features: {[feature_names[i] for i in feature_indices]}')

plot_partial_dependence(
    best_model, X_test,
    features=feature_indices,
    feature_names=feature_names
)

---
## 13. Classification Threshold Analysis

The default threshold of 0.5 may not be optimal for imbalanced data.  
We sweep thresholds and find the one that maximizes F1.

In [ ]:
print(f'Threshold analysis for: {best_name}')
optimal_threshold = threshold_analysis(best_model, X_test, y_test)

In [ ]:
# Compare default vs optimal threshold
from sklearn.metrics import classification_report as cr

y_prob = best_model.predict_proba(X_test)[:, 1]

print(f'=== Default Threshold (0.50) ===')
y_pred_default = (y_prob >= 0.50).astype(int)
print(cr(y_test, y_pred_default, target_names=label_names))

print(f'\n=== Optimal Threshold ({optimal_threshold:.2f}) ===')
y_pred_optimal = (y_prob >= optimal_threshold).astype(int)
print(cr(y_test, y_pred_optimal, target_names=label_names))

---
## 14. Conclusion

### Summary

| Step | Description |
|------|-------------|
| **Data** | 101,766 patient encounters from 130 US hospitals |
| **Target** | Binary — readmitted within 30 days (1) vs. not (0) |
| **Class Imbalance** | Handled via SMOTE on training data only |
| **Models** | Logistic Regression, Random Forest, XGBoost |
| **Evaluation** | Accuracy, Precision, Recall, F1, ROC-AUC |
| **Best Model** | Selected based on ROC-AUC + CV stability |
| **Explainability** | PFI and PDP applied on best model only |
| **Threshold** | Optimized for maximum F1 score |

### Key Findings
- Class imbalance (~11% positive) required SMOTE to enable meaningful model learning.
- ROC and Precision-Recall curves confirmed model ranking across all operating points.
- Permutation Feature Importance revealed the top clinical predictors of early readmission.
- Partial Dependence Plots showed the direction and shape of each feature's effect.
- Threshold tuning improved F1 beyond the default 0.5 cutoff.